In [ ]:
import dataclasses

import jax

from openpi.models import model as _model
from openpi.policies import droid_policy
from openpi.policies import policy_config as _policy_config
from openpi.shared import download
from openpi.training import config as _config
from openpi.training import data_loader as _data_loader

import Pyro5.api
import numpy as np

In [ ]:
config = _config.get_config("pi0_fast_droid")
checkpoint_dir = download.maybe_download("gs://openpi-assets/checkpoints/pi0_fast_droid")
print(f"Loading model from {checkpoint_dir}")


In [ ]:
# load the model
config = _config.get_config("pi0_RPM_low_mem_finetune")
checkpoint_dir = '/data/checkpoints/pi0_RPM_low_mem_finetune/LoRA_2GPU_pickblueblockblackbowl/4000'


# Create a trained policy.
policy = _policy_config.create_trained_policy(config, checkpoint_dir)

In [ ]:
# delete current policy to free up memory, then load a new one
del policy

In [ ]:
# Run inference on a dummy example. This example corresponds to observations produced by the DROID runtime.
example = droid_policy.make_droid_example()
result = policy.infer(example)
print("Actions shape:", result["actions"].shape)

In [ ]:
import pyrealsense2 as rs
import numpy as np
import cv2
import matplotlib.pyplot as plt



class RealSenseCamera:
    def __init__(self, serial_number, width=640, height=480, fps=30):
        self.serial = serial_number
        self.pipeline = rs.pipeline()
        self.config = rs.config()
        self.config.enable_device(self.serial)
        self.config.enable_stream(rs.stream.color, width, height, rs.format.bgr8, fps)
        self.pipeline.start(self.config)




    def get_image(self):
        frames = self.pipeline.wait_for_frames()
        color_frame = frames.get_color_frame()
        if not color_frame:
            return None

        bgr = np.asanyarray(color_frame.get_data())
        rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
        return rgb


    def release(self):
        self.pipeline.stop()




serial_head = "f1380660"
serial_wrist = "128422270284"


# # initialize cameras
cam_head = RealSenseCamera(serial_head)
cam_wrist = RealSenseCamera(serial_wrist)

In [ ]:
from openpi_client import image_tools

img_head = image_tools.resize_with_pad(
                        cam_head.get_image(), 224, 224
                    )

img_wrist = image_tools.resize_with_pad(
                        cam_wrist.get_image(), 224, 224
                    )


# combine the two images
combined = np.hstack((img_head, img_wrist))  # shape (224, 224*2, 3)

# display the combined image
plt.figure(figsize=(8, 4))
plt.imshow(combined)
plt.title("Head + Wrist(RGB 224x224)")
plt.axis('off')
plt.show()

In [ ]:
cam_head.release()
cam_wrist.release()


In [ ]:
img_left = image_tools.resize_with_pad(
                        cam_head.get_image(), 224, 224
                    )
img_wrist = image_tools.resize_with_pad(
                        cam_wrist.get_image(), 224, 224
                    )

observation = {
    'observation.image_scene': img_left,
    # "observation/exterior_image_1_left": np.zeros_like(img_left),  # dummy image for left
    'observation.image_wrist': img_wrist,
    # "observation/wrist_image_left": np.zeros_like(img_wrist),  # dummy image for wrist
    "observation.state": np.array([0]*13, dtype=np.float32),
    # "observation/gripper_position": np.array([0], dtype=np.float32),  
    # "observation/joint_position": np.zeros_like(obs["robot_state"], dtype=np.float32),
    # "observation/gripper_position": np.zeros_like([obs["gripper_state"]], dtype=np.float32),  
    "prompt": "pick the blue block and place it in the black bowl",
}

result = policy.infer(observation)

action_list = result["actions"]
print(action_list.shape)

In [ ]:
# vla_policy_client
import Pyro5.api
import numpy as np
import time
from openpi_client import image_tools

ns = Pyro5.api.locate_ns()  # Locate the name server
uri = ns.lookup("pi0_controller")  # Use the name server to look up the URI
controller = Pyro5.api.Proxy(uri)

# prompt
prompt = "pick up the blue block and place it into the black bowl"  # <-- change prompt

MAX_STEPS = 50000  # maximum number of steps to run
video_buffer = []  # buffer to store video frames

# dummy action
action = np.zeros((50,7), dtype=np.float32)
action_list = action.tolist()  # convert to list for sending

for step in range(MAX_STEPS):
    start = time.time()
    print(f"\n=== Step {step} ===")
    data_to_send = {
        "type": "action",
        "data": action_list,
        "step": step
    }
    start1 = time.time()
    print(f"Sending data to controller: {data_to_send}")
    obs = controller.step(data_to_send)
    end1 = time.time()
    print(f"Controller step took {end1 - start1:.2f} seconds")

    img_left = image_tools.resize_with_pad(
                            cam_head.get_image(), 224, 224
                        )
    img_wrist = image_tools.resize_with_pad(
                            cam_wrist.get_image(), 224, 224
                        )

    
    # save the images
    combined = np.hstack([img_left, img_wrist])
    video_buffer.append(combined)

    observation = {
        "observation.image_scene": img_left,
        # "observation/exterior_image_1_left": np.zeros_like(img_left),  # dummy image for left
        "observation.image_wrist": img_wrist,
        # "observation/wrist_image_left": np.zeros_like(img_wrist),  # dummy image for wrist
        "observation.state": np.array(obs["robot_state"], dtype=np.float32),
        # "observation/gripper_position": np.array([obs["gripper_state"]], dtype=np.float32),  
        # "observation/joint_position": np.zeros_like(obs["robot_state"], dtype=np.float32),
        # "observation/gripper_position": np.zeros_like([obs["gripper_state"]], dtype=np.float32),  
        "prompt": prompt,
    }

    result = policy.infer(observation)
    action_list = result["actions"].tolist()
    end2 = time.time()
    print(f"Inference took {end2 - end1:.2f} seconds")
    print(f"Action at step {step}:", action_list[0])
    end = time.time()
    print(f"Step {step} took {end - start:.2f} seconds")